# Qwen 3.5 27B - Colab Host with Local Access

Loads Qwen 3.5 27B (Q4_K_M GGUF) on a Colab L4 GPU and exposes an
OpenAI-compatible API via a Cloudflare tunnel.

## Steps:
1. Check GPU
2. Install dependencies
3. Download the model
4. Start the server
5. Start the tunnel
6. Use locally with opencode / curl / OpenAI SDK

## Troubleshooting:
- If the server won't start, run the **Debug** cell at the bottom.
- If you see `unknown model architecture: 'qwen35'`, your llama-cpp-python is too old. Re-run Cell 2 (it now installs the latest version).
- If you see CUDA errors in logs, the GPU build may have failed — try: `Runtime > Disconnect and delete runtime`, then re-run from Cell 1.
- If the model OOMs, reduce `CTX_SIZE` in Cell 4 (e.g. 2048).

In [ ]:
#@title 1. Check GPU
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('No GPU found. Enable GPU in Runtime > Change runtime type.')
else:
    print(result.stdout.strip())
    free_mb = int(result.stdout.strip().split(',')[2].strip().split()[0])
    if free_mb < 18000:
        print(f'WARNING: Only {free_mb}MB free. Model needs ~18GB. Consider resetting runtime.')
    else:
        print(f'GPU has {free_mb}MB free - should be sufficient.')

In [ ]:
#@title 2. Install Dependencies (with CUDA support)
# NOTE: This cell may appear to hang at 'Building wheels' for 5-10 minutes.
#       That's normal — it's compiling llama.cpp with CUDA support from source.
import os

# Must uninstall first if a previous version is installed
!pip uninstall -y llama-cpp-python 2>/dev/null

# Try pre-built CUDA wheel first (much faster), fall back to source build
print('Installing llama-cpp-python with CUDA support...')
print('This may take 5-10 minutes if building from source. Please wait.')

# Colab uses CUDA 12.x, try the pre-built cu121 wheels first
install_result = os.system(
    'pip install --upgrade "llama-cpp-python[server]" '
    '--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 '
    '2>/dev/null'
)

if install_result != 0:
    print('Pre-built wheel not available, building from source (this takes 5-10 minutes)...')
    os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'
    os.system('pip install --upgrade --no-cache-dir "llama-cpp-python[server]"')

!pip install --upgrade huggingface-hub

import llama_cpp
print(f'llama-cpp-python version: {llama_cpp.__version__}')

# Verify GPU support is available
try:
    test = llama_cpp.Llama
    print('llama_cpp.Llama class available: OK')
except Exception as e:
    print(f'ERROR: {e}')

# Quick sanity check: server module
import llama_cpp.server
print('Server module loaded OK')

In [ ]:
#@title 3. Download GGUF Model
GGUF_REPO = "unsloth/Qwen3.5-27B-GGUF" #@param {type:"string"}
GGUF_FILE = "Qwen3.5-27B-Q4_K_M.gguf" #@param {type:"string"}

from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id=GGUF_REPO,
    filename=GGUF_FILE,
    local_dir="/content/model",
)
print(f"Downloaded to: {model_path}")

In [ ]:
#@title 4. Start llama-cpp-python Server (OpenAI-compatible)
import subprocess, time, urllib.request

N_GPU_LAYERS = -1  # offload all layers to GPU
CTX_SIZE = 4096 #@param {type:"integer"}
PORT = 8000

# Kill any previous server on the same port
!lsof -ti :{PORT} | xargs kill -9 2>/dev/null || true

# Open log files instead of PIPE to avoid buffer deadlock
stdout_log = open("/content/server_stdout.log", "w")
stderr_log = open("/content/server_stderr.log", "w")

server_proc = subprocess.Popen(
    [
        "python", "-m", "llama_cpp.server",
        "--model", model_path,
        "--n_gpu_layers", str(N_GPU_LAYERS),
        "--n_ctx", str(CTX_SIZE),
        "--host", "0.0.0.0",
        "--port", str(PORT),
        "--alias", "default",
    ],
    stdout=stdout_log,
    stderr=stderr_log,
)

print(f"Server starting on port {PORT} (PID: {server_proc.pid})...")
print("Waiting for model to load (1-3 minutes on L4)...")

for i in range(90):  # up to 15 minutes
    time.sleep(10)
    # Check if process died
    if server_proc.poll() is not None:
        print(f"\nServer process exited with code {server_proc.returncode}!")
        print("Last 30 lines of stderr:")
        !tail -30 /content/server_stderr.log 2>/dev/null
        break
    # Show progress every 30s
    if i % 3 == 2:
        print("\n--- Recent server logs ---")
        !tail -5 /content/server_stderr.log 2>/dev/null
        print("---")
    try:
        resp = urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=5)
        if resp.status == 200:
            print(f"\nServer is healthy! (attempt {i+1}, {(i+1)*10}s)")
            break
    except Exception:
        print(f"  Waiting... ({(i+1)*10}s)", flush=True)
else:
    print("\nServer not ready after 15 minutes. Last 30 log lines:")
    !tail -30 /content/server_stderr.log 2>/dev/null
    print("\nRun the Debug cell below for more info.")

In [ ]:
#@title 5. Install & Start Cloudflare Tunnel
import subprocess, time

!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

PORT = 8000
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print("Starting tunnel...")

TUNNEL_URL = None
timeout_secs = 30
start = time.time()

while time.time() - start < timeout_secs:
    line = tunnel_proc.stderr.readline().decode(errors='replace')
    if not line and tunnel_proc.poll() is not None:
        print(f"Tunnel process exited unexpectedly with code {tunnel_proc.returncode}")
        break
    if 'https://' in line and '.trycloudflare.com' in line:
        for part in line.split():
            if part.startswith('https://') and '.trycloudflare.com' in part:
                TUNNEL_URL = part.rstrip('/')
                break
    if TUNNEL_URL:
        break

if TUNNEL_URL:
    print(f"\n{'='*60}")
    print(f"YOUR TUNNEL URL: {TUNNEL_URL}")
    print(f"{'='*60}")
    print(f"\nUse it as:")
    print(f"  base_url = '{TUNNEL_URL}/v1'")
else:
    print("\nTunnel URL not found within 30s. Possible issues:")
    print("  - cloudflared may have failed to install")
    print("  - network restrictions on this Colab runtime")
    print("\nTry re-running this cell.")

In [ ]:
#@title 6. Quick Test from Colab
import json, urllib.request

url = "http://localhost:8000/v1/chat/completions"
data = json.dumps({
    "model": "default",
    "messages": [{"role": "user", "content": "Hello! What model are you?"}],
    "max_tokens": 200,
}).encode()

req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(req, timeout=120) as resp:
        result = json.loads(resp.read())
        print("Response:")
        print(result["choices"][0]["message"]["content"])
except Exception as e:
    print(f"Request failed: {e}")
    print("\nCheck server logs:")
    !tail -20 /content/server_stderr.log 2>/dev/null

In [ ]:
#@title 7. Debug (run this if something goes wrong)
import subprocess

print("=== Server Process ===")
if 'server_proc' in globals():
    print(f"PID: {server_proc.pid}, Running: {server_proc.poll() is None}")
    if server_proc.poll() is not None:
        print(f"Exit code: {server_proc.returncode}")
else:
    print("No server_proc variable found. Has Cell 4 been run?")

print("\n=== Last 40 lines of server stderr ===")
!tail -40 /content/server_stderr.log 2>/dev/null || echo "No log file found"

print("\n=== Last 20 lines of server stdout ===")
!tail -20 /content/server_stdout.log 2>/dev/null || echo "No log file found"

print("\n=== GPU Memory ===")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv,noheader

print("\n=== Port 8000 Check ===")
!lsof -i :8000 2>/dev/null || echo "Nothing listening on port 8000"

print("\n=== Tunnel Process ===")
if 'tunnel_proc' in globals():
    print(f"PID: {tunnel_proc.pid}, Running: {tunnel_proc.poll() is None}")
else:
    print("No tunnel_proc variable found. Has Cell 5 been run?")

if 'TUNNEL_URL' in globals() and TUNNEL_URL:
    print(f"\nTunnel URL: {TUNNEL_URL}")
else:
    print("\nNo tunnel URL found.")

---
## Local Usage

After running all cells, copy the tunnel URL from Cell 5 output.

### Python (OpenAI SDK)
```python
from openai import OpenAI

client = OpenAI(
    base_url="YOUR_TUNNEL_URL/v1",
    api_key="not-needed",
)

response = client.chat.completions.create(
    model="default",
    messages=[{"role": "user", "content": "Hello!"}],
    max_tokens=200,
)
print(response.choices[0].message.content)
```

### cURL
```bash
curl YOUR_TUNNEL_URL/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "default", "messages": [{"role": "user", "content": "Hello!"}], "max_tokens": 200}'
```

### opencode
Create `opencode.json` in your project root:
```json
{
  "$schema": "https://opencode.ai/config.json",
  "provider": {
    "colab-qwen": {
      "npm": "@ai-sdk/openai-compatible",
      "name": "Qwen 3.5 27B (Colab)",
      "options": {
        "baseURL": "YOUR_TUNNEL_URL/v1"
      },
      "models": {
        "default": {
          "name": "Qwen 3.5 27B Q4_K_M",
          "limit": {
            "context": 4096,
            "output": 4096
          }
        }
      }
    }
  }
}
```
Replace `YOUR_TUNNEL_URL` with the URL from Cell 5 (e.g. `https://xxx-yyy-zzz.trycloudflare.com`).

Then in opencode: `/models` → select **Qwen 3.5 27B Q4_K_M**.

**Note:** The tunnel URL changes every time you restart the Colab runtime. You'll need to update `opencode.json` each time.
---